In [367]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import shapiro, mannwhitneyu, ttest_ind, levene
import warnings
warnings.filterwarnings('ignore')

# For VIF calculation (manual implementation without statsmodels)
from sklearn.linear_model import LinearRegression


In [368]:
# SECTION 1: LOAD ALL CLEANED DATASETS
prio = pd.read_csv('/Users/Marcy_Student/Desktop/Food Insecurity Analysis/datasets/cleaned_for_eda/cleaned_neighborhood_prioritization.csv')
efap = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/efap_cleaned.csv')
efap_nta_mapping = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/efap_nta_mapping.csv')
dim_map = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/dim_map.csv')
shelter_census = pd.read_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/shelter_census_clean.csv')

In [369]:
print(f"\n1. Neighborhood Prioritization: {prio.shape[0]} rows, {prio.shape[1]} columns")
print(f"2. EFAP Programs: {efap.shape[0]} rows, {efap.shape[1]} columns")
print(f"3. EFAP-NTA Mapping: {efap_nta_mapping.shape[0]} rows, {efap_nta_mapping.shape[1]} columns")
print(f"4. Dimension Map (Geography): {dim_map.shape[0]} rows, {dim_map.shape[1]} columns")
print(f"5. Shelter Census: {shelter_census.shape[0]} rows, {shelter_census.shape[1]} columns")


1. Neighborhood Prioritization: 197 rows, 11 columns
2. EFAP Programs: 561 rows, 7 columns
3. EFAP-NTA Mapping: 553 rows, 4 columns
4. Dimension Map (Geography): 262 rows, 7 columns
5. Shelter Census: 5204 rows, 6 columns


In [370]:
# Quick peek at each dataset
print("\n--- Dataset Previews ---")
print("\nPrioritization columns:", list(prio.columns))
print("EFAP columns:", list(efap.columns))
print("EFAP-NTA Mapping columns:", list(efap_nta_mapping.columns))
print("Dim Map columns:", list(dim_map.columns))
print("Shelter Census columns:", list(shelter_census.columns))


--- Dataset Previews ---

Prioritization columns: ['nta_id', 'nta_name', 'borough', 'food_insecure_percentage', 'food_insecure_percentage_rank', 'unemployment_rate', 'unemployment_rate_rank', 'vulnerable_population_percentage', 'vulnerable_population_percentage_rank', 'supply_gap', 'weighted_score']
EFAP columns: ['efap_id', 'program_name', 'access_type', 'has_pantry_access', 'has_kitchen_access', 'weekday_available', 'weekend_available']
EFAP-NTA Mapping columns: ['efap_id', 'nta_id', 'lat', 'lon']
Dim Map columns: ['nta_id', 'nta_name', 'cdta_id', 'cdta_name', 'boro_code', 'boro_name', 'the_geom_wkt']
Shelter Census columns: ['report_date', 'borough', 'community_districts', 'family_with_children_commercial_hotel', 'family_with_children_shelter', 'family_cluster']


In [371]:
# Merge EFAP with EFAP-NTA Mapping to get NTA for each site
efap_with_nta = efap.merge(efap_nta_mapping[['efap_id', 'nta_id']], on='efap_id', how='left')

In [372]:
efap_with_nta.head()


,efap_id,program_name,access_type,has_pantry_access,has_kitchen_access,weekday_available,weekend_available,nta_id
0,80604,HOLY APOSTLES SOUP KITCHEN,Kitchen,0,1,1,0,MN0401
1,85547,HOLY APOSTLES SOUP KITCHEN PANTRY,Pantry,1,0,1,0,MN0401
2,80757,ST. JOHN'S BREAD OF LIFE,Pantry,1,0,1,0,MN0501
3,85701,ARTISTS ATHLETES ACTIVISTS INCORPORATED,Pantry,1,0,1,0,MN0302
4,80546,DEWITT REFORMED CHURCH,Pantry,1,0,0,1,MN0302


In [373]:
print(f"EFAP sites with NTA mapping: {efap_with_nta['nta_id'].notna().sum()} of {len(efap_with_nta)}")
print(f"EFAP sites without NTA (will be excluded): {efap_with_nta['nta_id'].isna().sum()}")

EFAP sites with NTA mapping: 553 of 561
EFAP sites without NTA (will be excluded): 8


In [374]:
# Drop sites without NTA mapping
efap_with_nta = efap_with_nta.dropna(subset=['nta_id'])
print(f"EFAP sites after dropping unmapped: {len(efap_with_nta)}")

EFAP sites after dropping unmapped: 553


In [375]:
print("\n--- Step 2.2: Aggregate EFAP Sites to NTA Level ---")

nta_efap_agg = efap_with_nta.groupby('nta_id').agg(
    total_sites=('efap_id', 'count'),
    pantry_sites=('has_pantry_access', 'sum'),
    kitchen_sites=('has_kitchen_access', 'sum'),
    weekday_sites=('weekday_available', 'sum'),
    weekend_sites=('weekend_available', 'sum')
).reset_index()
print(f"Aggregated EFAP data at NTA level: {nta_efap_agg.shape[0]} NTAs")
nta_efap_agg


--- Step 2.2: Aggregate EFAP Sites to NTA Level ---
Aggregated EFAP data at NTA level: 152 NTAs


,nta_id,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites
0,BK0101,1,1,0,1,0
1,BK0102,1,1,0,1,0
2,BK0103,1,1,0,1,0
3,BK0104,1,1,0,1,0
4,BK0202,3,2,1,2,1
...,...,...,...,...,...,...
147,SI0201,1,1,0,1,0
148,SI0202,1,1,0,1,0
149,SI0203,1,1,1,1,0
150,SI0302,1,1,0,1,0


In [376]:
# Parse date and filter to 2023-2024
shelter_census['report_date'] = pd.to_datetime(shelter_census['report_date'])
shelter_census['year'] = shelter_census['report_date'].dt.year
shelter_census['month'] = shelter_census['report_date'].dt.month
shelter_census['Year-Month'] = shelter_census['report_date'].dt.to_period('M')

# Filter to 2023-2024
shelter_2023_2024 = shelter_census[shelter_census['year'].isin([2023, 2024])].copy()
print(f"Shelter records for 2023-2024: {len(shelter_2023_2024)}")

Shelter records for 2023-2024: 1436


In [377]:
# Fill NaN with 0 for shelter counts (assumption: NaN = no recorded families)
shelter_cols = ['family_with_children_commercial_hotel', 'family_with_children_shelter', 'family_cluster']
for col in shelter_cols:
    shelter_2023_2024[col] = shelter_2023_2024[col].fillna(0).astype(int)



In [378]:
# Aggregate to borough + CD level (sum over 2023-2024)
shelter_cd_agg = shelter_2023_2024.groupby(['borough', 'community_districts']).agg(
    total_families_with_children_shelter=('family_with_children_shelter', 'sum'),
).reset_index()

In [379]:
shelter_cd_agg

,borough,community_districts,total_families_with_children_shelter
0,Bronx,1.0,28947
1,Bronx,2.0,20047
2,Bronx,3.0,30888
3,Bronx,4.0,52319
4,Bronx,5.0,38815
5,Bronx,6.0,48601
6,Bronx,7.0,11174
7,Bronx,8.0,5714
8,Bronx,9.0,31536
9,Bronx,10.0,13473


In [380]:
# Create high shelter concentration flag (top 25%)
shelter_cd_agg['high_shelter_flag'] = (
    shelter_cd_agg['total_families_with_children_shelter'] >= shelter_cd_agg['total_families_with_children_shelter'].quantile(0.75)
).astype(int)

print(f"Community Districts with shelter data: {len(shelter_cd_agg)}")
print(f"High shelter concentration CDs: {shelter_cd_agg['high_shelter_flag'].sum()}")

Community Districts with shelter data: 59
High shelter concentration CDs: 15


In [381]:
# Extract CD number from cdta_id (e.g., 'BK14' -> 14)
dim_map['cd_number'] = dim_map['cdta_id'].str.extract(r'(\d+)').astype(float)

# Map borough codes to full names
boro_map = {'BK': 'Brooklyn', 'BX': 'Bronx', 'MN': 'Manhattan', 'QN': 'Queens', 'SI': 'Staten Island'}
dim_map['borough_from_cdta'] = dim_map['cdta_id'].str[:2].map(boro_map)

print(f"Unique NTAs in dim_map: {dim_map['nta_id'].nunique()}")
print(f"Unique CDTAs in dim_map: {dim_map['cdta_id'].nunique()}")

# Create NTA to CDTA lookup
nta_to_cdta = dim_map[['nta_id', 'cdta_id', 'cd_number', 'boro_name']].drop_duplicates()
print(f"\nNTA to CDTA mapping sample:")
print(nta_to_cdta.head(10))

Unique NTAs in dim_map: 262
Unique CDTAs in dim_map: 71

NTA to CDTA mapping sample:
   nta_id cdta_id  cd_number boro_name
0  BK0101    BK01        1.0  Brooklyn
1  BK0102    BK01        1.0  Brooklyn
2  BK0103    BK01        1.0  Brooklyn
3  BK0104    BK01        1.0  Brooklyn
4  BK0201    BK02        2.0  Brooklyn
5  BK0202    BK02        2.0  Brooklyn
6  BK0203    BK02        2.0  Brooklyn
7  BK0204    BK02        2.0  Brooklyn
8  BK0261    BK02        2.0  Brooklyn
9  BK0301    BK03        3.0  Brooklyn


In [382]:
# Merge shelter_cd_agg with nta_to_cdta
# Match on borough and community_districts (CD number)
shelter_cd_agg['community_districts'] = shelter_cd_agg['community_districts'].astype(float)

nta_shelter = nta_to_cdta.merge(
    shelter_cd_agg,
    left_on=['boro_name', 'cd_number'],
    right_on=['borough', 'community_districts'],
    how='left'
)

In [383]:
nta_efap_agg

,nta_id,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites
0,BK0101,1,1,0,1,0
1,BK0102,1,1,0,1,0
2,BK0103,1,1,0,1,0
3,BK0104,1,1,0,1,0
4,BK0202,3,2,1,2,1
...,...,...,...,...,...,...
147,SI0201,1,1,0,1,0
148,SI0202,1,1,0,1,0
149,SI0203,1,1,1,1,0
150,SI0302,1,1,0,1,0


In [384]:
nta_to_cdta

,nta_id,cdta_id,cd_number,boro_name
0,BK0101,BK01,1.0,Brooklyn
1,BK0102,BK01,1.0,Brooklyn
2,BK0103,BK01,1.0,Brooklyn
3,BK0104,BK01,1.0,Brooklyn
4,BK0201,BK02,2.0,Brooklyn
...,...,...,...,...
257,SI0391,SI03,3.0,Staten Island
258,SI9561,SI95,95.0,Staten Island
259,SI9591,SI95,95.0,Staten Island
260,SI9592,SI95,95.0,Staten Island


In [385]:
nta_shelter.head()

,nta_id,cdta_id,cd_number,boro_name,borough,community_districts,total_families_with_children_shelter,high_shelter_flag
0,BK0101,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
1,BK0102,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
2,BK0103,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
3,BK0104,BK01,1.0,Brooklyn,Brooklyn,1.0,0.0,0.0
4,BK0201,BK02,2.0,Brooklyn,Brooklyn,2.0,1708.0,0.0


In [386]:
nta_shelter[['nta_id', 'cdta_id', 'boro_name', 'cd_number', 
                            'total_families_with_children_shelter', 'high_shelter_flag']]

,nta_id,cdta_id,boro_name,cd_number,total_families_with_children_shelter,high_shelter_flag
0,BK0101,BK01,Brooklyn,1.0,0.0,0.0
1,BK0102,BK01,Brooklyn,1.0,0.0,0.0
2,BK0103,BK01,Brooklyn,1.0,0.0,0.0
3,BK0104,BK01,Brooklyn,1.0,0.0,0.0
4,BK0201,BK02,Brooklyn,2.0,1708.0,0.0
...,...,...,...,...,...,...
257,SI0391,SI03,Staten Island,3.0,0.0,0.0
258,SI9561,SI95,Staten Island,95.0,NaN,NaN
259,SI9591,SI95,Staten Island,95.0,NaN,NaN
260,SI9592,SI95,Staten Island,95.0,NaN,NaN


In [387]:
# Join Everything Together at NTA Level
# Start with prioritization as base (197 NTAs)
unified = prio.copy()

# Add EFAP aggregations (left join - some NTAs may have no sites)
unified = unified.merge(nta_efap_agg, left_on='nta_id', right_on='nta_id', how='left')

In [388]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,weighted_score,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,8.2210,1.0,1.0,0.0,1.0,0.0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,8.0704,3.0,3.0,0.0,3.0,0.0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,7.6866,4.0,3.0,1.0,1.0,3.0
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,7.3895,2.0,2.0,1.0,2.0,0.0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,7.2775,1.0,1.0,1.0,1.0,0.0


In [389]:
shelter_cd_agg.head()

,borough,community_districts,total_families_with_children_shelter,high_shelter_flag
0,Bronx,1.0,28947,1
1,Bronx,2.0,20047,1
2,Bronx,3.0,30888,1
3,Bronx,4.0,52319,1
4,Bronx,5.0,38815,1


In [390]:
shelter_cd_agg.head(20)

,borough,community_districts,total_families_with_children_shelter,high_shelter_flag
0,Bronx,1.0,28947,1
1,Bronx,2.0,20047,1
2,Bronx,3.0,30888,1
3,Bronx,4.0,52319,1
4,Bronx,5.0,38815,1
5,Bronx,6.0,48601,1
6,Bronx,7.0,11174,0
7,Bronx,8.0,5714,0
8,Bronx,9.0,31536,1
9,Bronx,10.0,13473,0


In [391]:
unified.isna().sum()

nta_id                                    0
nta_name                                  0
borough                                   0
food_insecure_percentage                  0
food_insecure_percentage_rank             0
unemployment_rate                         0
unemployment_rate_rank                    0
vulnerable_population_percentage          0
vulnerable_population_percentage_rank     0
supply_gap                                0
weighted_score                            0
total_sites                              45
pantry_sites                             45
kitchen_sites                            45
weekday_sites                            45
weekend_sites                            45
dtype: int64

### Assumption: Imputing Missing EFAP Site Counts as Zero

After joining the Neighborhood Prioritization dataset (197 NTAs) with the aggregated EFAP dataset (152 NTAs), 45 NTAs contained null values in EFAP-related columns (total_sites, pantry_site_count, kitchen_site_count, weekend_site_count, weekday_site_count). These null values emerged because those NTAs did not have corresponding records in the EFAP dataset.

We assume that these nulls do not represent missing or unreported data, but instead indicate that no EFAP sites are present in those neighborhoods. This interpretation is supported by the join structure: the prioritization dataset defines the universe of neighborhoods, and the EFAP dataset only contains neighborhoods where at least one food site exists. Hence, an unmatched NTA implies zero recorded EFAP supply.

Because of this, we impute these null EFAP values with 0 to accurately reflect the absence of food assistance sites rather than treat them as missing data. Dropping these rows would incorrectly remove valid neighborhoods from the analysis and bias results by excluding areas with potentially the lowest food coverage which are central to our research question.

This assumption allows us to preserve the full neighborhood universe while maintaining analytical consistency in coverage calculations.

In [392]:
unified['total_sites'] = unified['total_sites'].fillna(0).astype(int)
unified['pantry_sites'] = unified['pantry_sites'].fillna(0).astype(int)
unified['kitchen_sites'] = unified['kitchen_sites'].fillna(0).astype(int)
unified['weekday_sites'] = unified['weekday_sites'].fillna(0).astype(int)
unified['weekend_sites'] = unified['weekend_sites'].fillna(0).astype(int)

In [393]:
# Add shelter data (via NTA-CDTA mapping)
unified = unified.merge(
    nta_shelter[['nta_id', 'cdta_id', 'total_families_with_children_shelter', 'high_shelter_flag']],
    on='nta_id',
    how='left'
)

In [394]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,weighted_score,total_sites,pantry_sites,kitchen_sites,weekday_sites,weekend_sites,cdta_id,total_families_with_children_shelter,high_shelter_flag
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,8.2210,1,1,0,1,0,BK01,0.0,0.0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,8.0704,3,3,0,3,0,BX05,38815.0,1.0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,7.6866,4,3,1,1,3,BX09,31536.0,1.0
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,7.3895,2,2,1,2,0,MN12,0.0,0.0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,7.2775,1,1,1,1,0,BK15,7847.0,0.0


In [395]:
unified.shape

(197, 19)

In [396]:
unified.isnull().sum()

nta_id                                   0
nta_name                                 0
borough                                  0
food_insecure_percentage                 0
food_insecure_percentage_rank            0
unemployment_rate                        0
unemployment_rate_rank                   0
vulnerable_population_percentage         0
vulnerable_population_percentage_rank    0
supply_gap                               0
weighted_score                           0
total_sites                              0
pantry_sites                             0
kitchen_sites                            0
weekday_sites                            0
weekend_sites                            0
cdta_id                                  0
total_families_with_children_shelter     0
high_shelter_flag                        0
dtype: int64

## 3: FEATURE ENGINEERING - COVERAGE RATIO

COVERAGE RATIO DEFINITION:
We want to measure whether food assistance supply aligns with neighborhood need.

Formula Options:
1. coverage_ratio = total_sites / food_insecure_percentage
   - Interpretation: Sites per unit of food insecurity rate
   - Higher = better coverage relative to need

2. sites_per_1000_insecure = (total_sites / food_insecure_percentage) * 100
   - Scaled version for easier interpretation

3. Binary: has_coverage_gap = 1 if coverage_ratio < median, else 0

We'll use food_insecure_percentage as the denominator because:
- It directly measures food insecurity at the neighborhood level
- It's available for all 197 NTAs
- It's a key component of the prioritization score

In [397]:
# Create coverage ratio
# Avoid division by zero (though all NTAs have food insecurity > 0)
unified['coverage_ratio'] = unified['total_sites'] / unified['food_insecure_percentage']

In [398]:
print(unified['coverage_ratio'].describe())


count    197.000000
mean       0.202451
std        0.260725
min        0.000000
25%        0.041135
50%        0.124844
75%        0.295276
max        2.054795
Name: coverage_ratio, dtype: float64


In [399]:
# Create Priority Groups
# Binary high priority (top 25%)
unified['is_high_priority'] = (
    unified['weighted_score'] >= unified['weighted_score'].quantile(0.75)
).astype(int)

In [400]:
print(f"\nHigh Priority NTAs (top 25% by weighted_score): {unified['is_high_priority'].sum()}")



High Priority NTAs (top 25% by weighted_score): 50


In [401]:
# Create Coverage Categories for Logistic Regression

# Binary outcome: Low coverage vs High coverage
# Using median as cutoff
coverage_median = unified['coverage_ratio'].median()
unified['coverage_category'] = (unified['coverage_ratio'] >= coverage_median).astype(int)
unified['coverage_category_label'] = unified['coverage_category'].map({0: 'Low', 1: 'High'})

print(f"Coverage Ratio Median: {coverage_median:.4f}")
print(f"Low Coverage NTAs: {(unified['coverage_category'] == 0).sum()}")
print(f"High Coverage NTAs: {(unified['coverage_category'] == 1).sum()}")

Coverage Ratio Median: 0.1248
Low Coverage NTAs: 98
High Coverage NTAs: 99


- Additional Features

In [402]:
# Has any kitchen access
unified['has_kitchen'] = (unified['kitchen_sites'] > 0).astype(int)

# Has any weekend access
unified['has_weekend'] = (unified['weekend_sites'] > 0).astype(int)

# Log-transformed features (for modeling, avoiding log(0))
unified['log_total_sites'] = np.log1p(unified['total_sites'])
unified['log_coverage_ratio'] = np.log1p(unified['coverage_ratio'])

In [403]:
# Normalized priority score (min-max scaling)
unified['priority_normalized'] = (
    (unified['weighted_score'] - unified['weighted_score'].min()) / 
    (unified['weighted_score'].max() - unified['weighted_score'].min())
)

In [404]:
# create two new features for logistic regression low coverage vs high coverage
unified['is_low_coverage'] = (unified['coverage_category'] == 0).astype(int)
unified['is_high_coverage'] = (unified['coverage_category'] == 1).astype(int)

In [405]:
unified.columns

Index(['nta_id', 'nta_name', 'borough', 'food_insecure_percentage',
       'food_insecure_percentage_rank', 'unemployment_rate',
       'unemployment_rate_rank', 'vulnerable_population_percentage',
       'vulnerable_population_percentage_rank', 'supply_gap', 'weighted_score',
       'total_sites', 'pantry_sites', 'kitchen_sites', 'weekday_sites',
       'weekend_sites', 'cdta_id', 'total_families_with_children_shelter',
       'high_shelter_flag', 'coverage_ratio', 'is_high_priority',
       'coverage_category', 'coverage_category_label', 'has_kitchen',
       'has_weekend', 'log_total_sites', 'log_coverage_ratio',
       'priority_normalized', 'is_low_coverage', 'is_high_coverage'],
      dtype='object')

In [406]:
unified.shape

(197, 30)

In [407]:
unified.head()

,nta_id,nta_name,borough,food_insecure_percentage,food_insecure_percentage_rank,unemployment_rate,unemployment_rate_rank,vulnerable_population_percentage,vulnerable_population_percentage_rank,supply_gap,...,is_high_priority,coverage_category,coverage_category_label,has_kitchen,has_weekend,log_total_sites,log_coverage_ratio,priority_normalized,is_low_coverage,is_high_coverage
0,BK0104,East Williamsburg,Brooklyn,35.99,1,6.38,126,12.43,146,2.776626e+06,...,1,0,Low,0,0,0.693147,0.027406,1.000000,1,0
1,BX0501,University Heights (South)-Morris Heights,Bronx,29.44,14,11.98,20,19.63,34,1.669389e+06,...,1,0,Low,0,0,1.386294,0.097038,0.977019,1,0
2,BX0901,Soundview-Bruckner-Bronx River,Bronx,22.63,36,10.06,32,21.43,25,1.625976e+06,...,1,1,High,1,1,1.609438,0.162762,0.918451,0,1
3,MN1202,Washington Heights (North),Manhattan,24.29,28,12.25,19,18.57,41,1.463457e+06,...,1,0,Low,1,0,1.098612,0.079124,0.873113,1,0
4,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,Brooklyn,21.11,42,4.91,170,15.95,81,1.907056e+06,...,1,0,Low,1,0,0.693147,0.046283,0.856022,1,0


In [408]:
#export unified dataset for modeling
unified.to_csv('/Users/Marcy_Student/Desktop/Marcy_Projects/CID_Food_Access/data/clean/unified_dataset_for_modeling.csv', index=False)

### Next Step:
- Statistical Analysis and Modeling